# IFRS 9 staging, scenarios, overlays, and reconciliation

Use the original IFRS 9 package on a contractual-period teaching schedule. Staging policy, marginal PD, scenario weighting, overlays, and ledger reconciliation remain separate and visible.

All data are generated locally unless this notebook explicitly calls a reviewed adapter. Results are educational and require independent validation before any real use.

In [ ]:
import numpy as np
import pandas as pd

from creditriskbook.data import load_case_dataset
from creditriskbook.ifrs9 import (
    Scenario, StagingPolicy, apply_overlay, assign_stages,
    calculate_ecl, reconcile_ecl,
)

schedule = load_case_dataset("synthetic_ifrs9_schedule", n_rows=120, seed=909).frame
accounts = pd.DataFrame({
    "account_id": ["A", "B", "C", "D"],
    "origination_pd_12m": [0.01, 0.01, 0.02, 0.03],
    "current_pd_12m": [0.012, 0.035, 0.04, 0.30],
    "days_past_due": [0, 0, 45, 95],
    "watchlist_flag": [False, False, False, False],
    "default_flag": [False, False, False, True],
})
staged = assign_stages(accounts, StagingPolicy())
assert staged["stage"].tolist() == [1, 2, 2, 3]
print(staged[["account_id", "stage", "stage_reason", "pd_ratio"]])

In [ ]:
scenarios = (
    Scenario("upside", 0.20, pd_multiplier=0.80, lgd_multiplier=0.90),
    Scenario("base", 0.55),
    Scenario("downside", 0.25, pd_multiplier=1.50, lgd_multiplier=1.20, ead_multiplier=1.05),
)
result = calculate_ecl(schedule, scenarios)
assert np.allclose(result.reconciliation["amount"], result.account["ecl"].sum())
print(result.reconciliation)
print(result.account.groupby("stage")["ecl"].agg(["count", "sum"]))

In [ ]:
selected = result.account.head(2)[["account_id"]].copy()
selected["overlay_type"] = ["additive", "multiplicative"]
selected["overlay_value"] = [25.0, 1.10]
selected["overlay_reason"] = ["bounded data gap", "bounded scenario gap"]
adjusted = apply_overlay(result.account, selected)
ledger_total = float(adjusted["post_overlay_ecl"].sum())
reconciliation = reconcile_ecl(adjusted, ledger_total=ledger_total)
assert reconciliation["within_tolerance"]
print(adjusted.head())
print(reconciliation)

## Control boundary

These are educational calculations. A real close requires an approved IFRS 9 accounting policy, controlled perimeter, scenario governance, independent validation, overlay approval, ledger posting controls, and disclosure review.